In [1]:
# CELL 1 — INSTALL DEPENDENCIES
# Run this cell first on Kaggle

!pip install ultralytics --quiet
!pip install scipy --quiet
!pip install opencv-python-headless --quiet
!pip install matplotlib tqdm --quiet

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00a 0:00:01
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
# CELL 2 — IMPORTS & GLOBAL SETUP

import os, sys, json, shutil, random, time, math
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from scipy.special import lambertw

from ultralytics import YOLO

print("[OK] All imports successful.")
print(f"[INFO] Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

[OK] All imports successful.
[INFO] Device: cuda


In [4]:
# CELL 3 — ███████████  USER CONFIG SECTION  ███████████

# ── Paths ──────────────────────────────────────────────────────────────────
DATASET_ROOT = "/kaggle/input/datasets/bichhanhluuthi/my-dataset/My_Thesis_Dataset"      
REPO_ROOT    = "/kaggle/working/AO-Exp-Attack"        

# ── Video target ───────────────────────────────────────────────────────────
VIDEO_TARGET = "gostiny_dvor_001"
# Options:
#   annichkov_most_001 | annichkov_most_002 | dvorzovy_most_001
#   gostiny_dvor_001   | gostiny_dvor_002   | zagorodny_proezd_001
#   universal

# ── Class target ───────────────────────────────────────────────────────────
CLASS_MODE = "vehicle"       # "person" or "vehicle"

# ── Attack hyperparameters ─────────────────────────────────────────────────
LAMBDA1      = 0.08         # λ₁: Nuclear-norm regularization weight
LAMBDA2      = 0.0001       # λ₂: Frobenius-norm regularization weight
EPSILON      = 24.0         # Perturbation budget (used for norm clipping)
LR_INIT      = 1.0          # η₀: Initial step-size for AO-Exp
N_ITER       = 2000         # T: Total number of gradient iterations
K_RANK       = 20           # k: Number of singular values retained (top-k)
TAU          = 0.25         # τ: Confidence threshold for mask generation
ALPHA        = 3.0          # α: Weight for the foreground loss
GAMMA        = 3.0          # γ: Weight for the confidence loss
BETA         = 0.5          # β: Weight for the background loss
BATCH_FRAMES = 8            # B: Number of frames sampled per iteration
MAX_FRAMES   = 1200         # Maximum number of frames loaded per video

# ── YOLOv11m ──────────────────────────────────────────────────────────────
YOLO_MODEL   = "yolo11m.pt"   
IMG_SIZE     = 640

# ── Dataset split ─────────────────────────────────────────────────────────
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15

# ── Misc ──────────────────────────────────────────────────────────────────
SEED   = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[CONFIG] VIDEO_TARGET = {VIDEO_TARGET}")
print(f"[CONFIG] CLASS_MODE   = {CLASS_MODE}")
print(f"[CONFIG] DEVICE       = {DEVICE}")
print(f"[CONFIG] N_ITER={N_ITER} | K_RANK={K_RANK} | λ₁={LAMBDA1} | λ₂={LAMBDA2}")

[CONFIG] VIDEO_TARGET = gostiny_dvor_001
[CONFIG] CLASS_MODE   = vehicle
[CONFIG] DEVICE       = cuda
[CONFIG] N_ITER=2000 | K_RANK=20 | λ₁=0.08 | λ₂=0.0001


In [5]:
# CELL 4 — SEED & DIRECTORY SETUP

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── 6 video names ─────────────────────────────────────────────────────────
VIDEO_NAMES = [
    "annichkov_most_001",
    "annichkov_most_002",
    "dvorzovy_most_001",
    "gostiny_dvor_001",
    "gostiny_dvor_002",
    "zagorodny_proezd_001",
]

# ── Class mapping: CLASS_MODE → set of YOLOv11m COCO class IDs ────────────
# YOLOv11m pretrained COCO (80 classes, 0-indexed):
#   person = 0
#   car=2, bus=5, truck=7
YOLO_PERSON_IDS  = {0}
YOLO_VEHICLE_IDS = {2, 5, 7}

CLASS_MODE_TO_YOLO = {
    "person":  YOLO_PERSON_IDS,
    "vehicle": YOLO_VEHICLE_IDS,
}

JSON_TO_YOLO_LABEL = {
    2: 0,  # JSON category_id=2 (person) → YOLO local class 0
    1: 1,  # JSON category_id=1 (vehicle) → YOLO local class 1
}
YOLO_LABEL_NAMES = ["person", "vehicle"]  # names[0], names[1] trong data.yaml

# ── Create output directories ─────────────────────────────────────────────
DATASET_DIR  = Path(REPO_ROOT) / "dataset_yolo"
OUTPUT_DIR   = Path(REPO_ROOT) / "outputs"
PERTURB_DIR  = Path(REPO_ROOT) / "perturbations"
VIZ_DIR      = Path(REPO_ROOT) / "visualizations"

for d in [DATASET_DIR, OUTPUT_DIR, PERTURB_DIR, VIZ_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("[OK] Seed set. Directories ready:")
for d in [DATASET_DIR, OUTPUT_DIR, PERTURB_DIR, VIZ_DIR]:
    print(f"      {d}")

[OK] Seed set. Directories ready:
      /kaggle/working/AO-Exp-Attack/dataset_yolo
      /kaggle/working/AO-Exp-Attack/outputs
      /kaggle/working/AO-Exp-Attack/perturbations
      /kaggle/working/AO-Exp-Attack/visualizations


In [6]:
# CELL 5 — DATASET PREPARATION
# Convert COCO JSON annotations to YOLO label format
# Split data 70/15/15 per video and create separate test_{video_name} directories

def coco_bbox_to_yolo(bbox, img_w, img_h):
    """[x, y, w, h] tuyệt đối → [cx, cy, w, h] chuẩn hóa theo ảnh."""
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    w  = w / img_w
    h  = h / img_h
    cx = max(0.0, min(1.0, cx))
    cy = max(0.0, min(1.0, cy))
    w  = max(0.0, min(1.0, w))
    h  = max(0.0, min(1.0, h))
    return cx, cy, w, h

def build_yolo_dataset(dataset_root: str, output_dir: Path,
                       train_r: float = 0.70, val_r: float = 0.15):
    dataset_root = Path(dataset_root)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    splits = {"train": [], "val": [], "test": []}
    video_test_files = {}

    for video_name in VIDEO_NAMES:
        vid_dir = dataset_root / video_name
        ann_path = vid_dir / "annotations.json"
        img_dir  = vid_dir / "images"
        
        if not ann_path.exists():
            alt_ann = dataset_root / "annotations" / f"{video_name}.json"
            if alt_ann.exists(): ann_path = alt_ann
        if not img_dir.exists():
            alt_img = dataset_root / "images" / video_name
            if alt_img.exists(): img_dir = alt_img
            
        if not ann_path.exists() or not img_dir.exists():
            print(f"  [WARN] Thiếu dữ liệu cho {video_name} → skip")
            continue

        with open(ann_path, "r") as f:
            coco_data = json.load(f)

        # Fast indexing: image_id → (width, height, filename) & annotations
        img_idx = {img["id"]: (img["width"], img["height"], img["file_name"]) 
                   for img in coco_data.get("images", [])}
        ann_idx = {iid: [] for iid in img_idx}
        for ann in coco_data.get("annotations", []):
            if ann["image_id"] in ann_idx:
                ann_idx[ann["image_id"]].append(ann)

        # Create output directories for the current video
        v_img_dir = output_dir / video_name / "images"
        v_lbl_dir = output_dir / video_name / "labels"
        v_img_dir.mkdir(parents=True, exist_ok=True)
        v_lbl_dir.mkdir(parents=True, exist_ok=True)

        paired = []
        for iid, (iw, ih, fname) in img_idx.items():
            stem = Path(fname).stem
            found_img = None
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                p = img_dir / f"{stem}{ext}"
                if p.exists():
                    found_img = p
                    break
            if not found_img: continue

            # Copy the image file
            out_img = v_img_dir / f"{stem}.jpg"
            if not out_img.exists(): shutil.copy2(found_img, out_img)

            # Write the YOLO label file
            out_lbl = v_lbl_dir / f"{stem}.txt"
            lines = []
            for ann in ann_idx.get(iid, []):
                cid = ann.get("category_id")
                if cid not in JSON_TO_YOLO_LABEL: continue  
                
                yolo_cls = JSON_TO_YOLO_LABEL[cid]
                cx, cy, bw, bh = coco_bbox_to_yolo(ann["bbox"], iw, ih)
                lines.append(f"{yolo_cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            
            out_lbl.write_text("\n".join(lines))
            paired.append((str(out_img), str(out_lbl)))

        # Perform 70/15/15 data split
        random.shuffle(paired)
        n = len(paired)
        n_train, n_val = int(n * train_r), int(n * val_r)
        train_p, val_p, test_p = paired[:n_train], paired[n_train:n_train+n_val], paired[n_train+n_val:]
        
        splits["train"].extend(train_p)
        splits["val"].extend(val_p)
        splits["test"].extend(test_p)
        video_test_files[video_name] = test_p
        print(f"  [{video_name}] train={len(train_p)} | val={len(val_p)} | test={len(test_p)}")

    # Write global split files
    for sname, pairs in splits.items():
        (output_dir / f"{sname}.txt").write_text("\n".join(p[0] for p in pairs))

    # Copy the test set for each video separately (used for UAP evaluation)
    for vname, test_pairs in video_test_files.items():
        vtest = output_dir / f"test_{vname}"
        (vtest/"images").mkdir(parents=True, exist_ok=True)
        (vtest/"labels").mkdir(parents=True, exist_ok=True)
        for img_p, lbl_p in test_pairs:
            dst_i = vtest/"images"/Path(img_p).name
            dst_l = vtest/"labels"/Path(lbl_p).name
            if not dst_i.exists(): shutil.copy2(img_p, dst_i)
            if Path(lbl_p).exists() and not dst_l.exists(): shutil.copy2(lbl_p, dst_l)

    # Write the standard YOLO data.yaml configuration file
    (output_dir / "data.yaml").write_text(
        f"path: {output_dir}\n"
        f"train: train.txt\nval: val.txt\ntest: test.txt\n"
        f"nc: 2\nnames: {YOLO_LABEL_NAMES}\n"
    )
    
    total = {k: len(v) for k, v in splits.items()}
    print(f"\n[DATA] Done → train={total['train']} | val={total['val']} | test={total['test']}")
    return output_dir, video_test_files

print("[OK] Dataset preparation function ready.")

[OK] Dataset preparation function ready.


In [7]:
# CELL 6 — VIDEO FRAME DATASET & LOADER
# Convert COCO JSON annotations to YOLO label format
# Split data 70/15/15 per video and create separate test_{video_name} directories

class VideoFrameDataset(torch.utils.data.Dataset):
    """
    Load frames từ thư mục ảnh.
    QUAN TRỌNG: frames phải được sort theo tên file (= chronological order)
    vì dataset đặt tên theo timeline.
    Trả về tensor [0,1] shape (C,H,W).
    """

    def __init__(self, video_dir: str,
                 img_size:    int  = 640,
                 max_frames:  int  = 300,
                 sample_mode: str  = "uniform",   # "uniform" | "all" | "random"
                 stride_cap:  int  = 50):          # Maximum allowed distance between two consecutive frames
        """
        sample_mode:
            "uniform"  → Select max_frames evenly spaced across the entire timeline (RECOMMENDED).
            "all"      → Load all frames (be cautious with long videos, e.g., 3607 frames, as it consumes high RAM).
            "random"   → Shuffle and select max_frames (loses temporal order).
        stride_cap:
            For "uniform" mode, if the calculated stride exceeds stride_cap, max_frames is increased
            to ensure that too many consecutive frames are not skipped.
        """
        self.img_size = img_size

        # ── Locate all frames and sort by filename (chronological order) ────────
        exts  = (".jpg", ".jpeg", ".png", ".bmp")
        all_paths = []
        for e in exts:
            all_paths += list(Path(video_dir).rglob(f"*{e}"))

        if not all_paths:
            raise ValueError(f"No frames found in: {video_dir}")

        # Sort by filename to ensure chronological order
        all_paths = sorted(all_paths, key=lambda p: p.name)
        N = len(all_paths)

        # ── Stratified temporal sampling ───────────────────────────────────
        if sample_mode == "all" or max_frames >= N:
            self.paths = all_paths
            strategy = f"all {N} frames"

        elif sample_mode == "uniform":
            # Select max_frames evenly spaced points in the range [0, N-1]
            indices    = self._uniform_indices(N, max_frames)
            self.paths = [all_paths[i] for i in indices]
            actual_stride = indices[1] - indices[0] if len(indices) > 1 else 1
            strategy = (f"uniform {len(self.paths)}/{N} frames "
                        f"(stride≈{actual_stride})")

        elif sample_mode == "random":
            idxs       = sorted(random.sample(range(N), min(max_frames, N)))
            self.paths = [all_paths[i] for i in idxs]
            strategy = f"random {len(self.paths)}/{N} frames"

        else:
            raise ValueError(f"Unknown sample_mode: {sample_mode}")

        print(f"    → Sampling: {strategy}")
        print(f"       First: {self.paths[0].name}  "
              f"Last: {self.paths[-1].name}")

    @staticmethod
    def _uniform_indices(N: int, k: int) -> List[int]:
        """
        Return k evenly spaced indices within the range [0, N-1].
        Ensures that both the first and the last frames are always included.
        """
        if k >= N:
            return list(range(N))
        if k == 1:
            return [N // 2]
        # linspace → round → unique → sort
        indices = sorted(set(
            int(round(i * (N - 1) / (k - 1))) for i in range(k)
        ))
    
        while len(indices) < k and len(indices) < N:
            gaps = [(indices[i+1] - indices[i], i)
                    for i in range(len(indices)-1)]
            _, max_i = max(gaps)
            mid = (indices[max_i] + indices[max_i+1]) // 2
            indices.insert(max_i + 1, mid)
        return indices[:k]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.paths[idx]))
        if img is None:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        img    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img    = cv2.resize(img, (self.img_size, self.img_size))
        tensor = torch.from_numpy(img).permute(2,0,1).float() / 255.0
        return tensor, str(self.paths[idx])


# ── Optimal frame count based on the actual number of frames in the video ──
def recommend_max_frames(total_frames: int) -> int:
    """
    annichkov_most_001:  914  → ~300
    annichkov_most_002: 3607  → ~500
    dvorzovy_most_001:  1158  → ~350
    gostiny_dvor_001:   1926  → ~400
    gostiny_dvor_002:   2461  → ~450
    zagorodny_proezd_001: 912 → ~300
    """
    if total_frames <= 1000:
        return min(total_frames, 300)
    elif total_frames <= 2000:
        return 400
    elif total_frames <= 3000:
        return 450
    else:
        return 500


# ── Recommended max_frames table for each video ────────────────────────────
VIDEO_FRAME_COUNTS = {
    "annichkov_most_001":   914,
    "annichkov_most_002":  3607,
    "dvorzovy_most_001":   1158,
    "gostiny_dvor_001":    1926,
    "gostiny_dvor_002":    2461,
    "zagorodny_proezd_001": 912,
}

VIDEO_MAX_FRAMES = {
    v: recommend_max_frames(n)
    for v, n in VIDEO_FRAME_COUNTS.items()
}

print("[OK] VideoFrameDataset (temporal stratified sampling) defined.")
print("\n[INFO] Recommended max_frames per video:")
for v, mf in VIDEO_MAX_FRAMES.items():
    total = VIDEO_FRAME_COUNTS[v]
    stride = total // mf
    print(f"  {v:<30}  total={total:>4}  max_frames={mf}  "
          f"stride≈{stride}  coverage=100%")


def load_frames_for_video(video_dir:   str,
                           img_size:    int  = 640,
                           max_frames:  int  = 300,
                           sample_mode: str  = "uniform",
                           batch_size:  int  = 4,
                           device: torch.device = torch.device("cpu")
                           ) -> List[Tensor]:
    """
    Load frames from video_dir using stratified temporal sampling.

    Args:
        video_dir   : Path to the directory containing frames (named chronologically).
        img_size    : Resize frames to (img_size × img_size).
        max_frames  : Maximum number of frames to extract (evenly distributed across the timeline).
        sample_mode : "uniform" (recommended) | "all" | "random".
        batch_size  : Batch size for the DataLoader.
        device      : Target device ('cuda' or 'cpu').

    Returns:
        List of tensors with shape (C, H, W) normalized to [0, 1].
    """
    print(f"  [LOADER] Loading from: {Path(video_dir).name}")
    ds = VideoFrameDataset(
        video_dir   = video_dir,
        img_size    = img_size,
        max_frames  = max_frames,
        sample_mode = sample_mode,
    )

    loader = torch.utils.data.DataLoader(
        ds,
        batch_size  = batch_size,
        shuffle     = False,       
        num_workers = 2,
        pin_memory  = torch.cuda.is_available(),
    )

    frames = []
    for batch_imgs, _ in loader:
        for i in range(batch_imgs.shape[0]):
            frames.append(batch_imgs[i].to(device))

    print(f"    → Loaded {len(frames)} frames | "
          f"device={device} | img_size={img_size}px")
    return frames


def _create_demo_dataset(base_dir: Path, n_frames: int = 40) -> Path:
    """
    Create a synthetic dataset with chronologically named frames
    (frame_0000.jpg → frame_0039.jpg) for pipeline testing.
    """
    base_dir.mkdir(parents=True, exist_ok=True)
    for vname in VIDEO_NAMES:
        img_dir = base_dir / f"test_{vname}" / "images"
        lbl_dir = base_dir / f"test_{vname}" / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        for i in range(n_frames):
            arr = (np.random.rand(480, 640, 3) * 255).astype(np.uint8)
            cv2.putText(arr, f"{vname[:10]} f{i:04d}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                        0.7, (255,255,255), 2)
            cv2.imwrite(str(img_dir / f"frame_{i:04d}.jpg"), arr)
            cx = 0.3 + 0.4 * (i / n_frames)   
            (lbl_dir / f"frame_{i:04d}.txt").write_text(
                f"0 {cx:.4f} 0.50 0.15 0.35\n")

    for split in ["train", "val", "test"]:
        imgs = []
        for vname in VIDEO_NAMES:
            imgs += sorted(str(p) for p in
                (base_dir / f"test_{vname}" / "images").glob("*.jpg"))
        (base_dir / f"{split}.txt").write_text("\n".join(imgs))

    (base_dir / "data.yaml").write_text(
        f"path: {base_dir}\ntrain: train.txt\nval: val.txt\n"
        f"test: test.txt\nnc: 2\nnames: ['person','vehicle']\n")

    print(f"[DEMO] Synthetic dataset → {base_dir}  "
          f"({n_frames} frames/video, chronological naming)")
    return base_dir

def load_frames_from_path_list(path_list: List[str], img_size: int, max_frames: int, 
                               device: torch.device, sample_mode: str = "uniform") -> List[Tensor]:
    if not path_list: return []
    path_list = sorted(path_list, key=lambda p: Path(p).name)
    N = len(path_list)
    if sample_mode == "uniform" and max_frames < N:
        indices = VideoFrameDataset._uniform_indices(N, max_frames)
    else:
        indices = list(range(min(max_frames, N)))
        
    frames = []
    for idx in indices:
        img = cv2.imread(path_list[idx])
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        frames.append(torch.from_numpy(img).permute(2,0,1).float() / 255.0)
    return [f.to(device) for f in frames]

print("\n[OK] load_frames_for_video + demo dataset defined.")

[OK] VideoFrameDataset (temporal stratified sampling) defined.

[INFO] Recommended max_frames per video:
  annichkov_most_001              total= 914  max_frames=300  stride≈3  coverage=100%
  annichkov_most_002              total=3607  max_frames=500  stride≈7  coverage=100%
  dvorzovy_most_001               total=1158  max_frames=400  stride≈2  coverage=100%
  gostiny_dvor_001                total=1926  max_frames=400  stride≈4  coverage=100%
  gostiny_dvor_002                total=2461  max_frames=450  stride≈5  coverage=100%
  zagorodny_proezd_001            total= 912  max_frames=300  stride≈3  coverage=100%

[OK] load_frames_for_video + demo dataset defined.


In [8]:
# CELL 7 — FROZEN YOLOv11m MODEL WRAPPER
# Freeze all weights, only compute gradient w.r.t. input perturbation δ

class FrozenYOLO(nn.Module):

    def __init__(self, model_name: str = "yolo11m.pt",
                 device: torch.device = torch.device("cpu"),
                 img_size:        int   = 640,
                 target_classes:  set   = None,
                 conf_threshold:  float = 0.25,
                 iou_threshold:   float = 0.45):
        super().__init__()
        self.img_size       = img_size
        self.target_classes = target_classes or {0}
        self.conf_threshold = conf_threshold
        self.iou_threshold  = iou_threshold
        self.device         = device

        # Load high-level YOLO (cho NMS/boxes) & internal nn.Module (cho grad)
        self._yolo_hl = YOLO(model_name)
        self.model    = self._yolo_hl.model.to(device)

        # Freeze ALL weights
        for p in self.model.parameters():
            p.requires_grad_(False)
        
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"  [YOLO] Loaded {model_name} | params={n_params:,} "
              f"| target_cls={self.target_classes}")

        # 1. Dummy forward để YOLO khởi tạo buffers (anchors, strides, DFL)
        dummy = torch.zeros(1, 3, self.img_size, self.img_size, device=self.device)
        with torch.no_grad():
            self.model(dummy)
            
        # 2. Clone các buffer quan trọng để chuyển thành tensor thường
        for m in self.model.modules():
            m_name = type(m).__name__
            if m_name == 'Detect':
                if hasattr(m, 'anchors') and m.anchors is not None:
                    m.anchors = m.anchors.clone()
                if hasattr(m, 'strides') and m.strides is not None:
                    m.strides = m.strides.clone()
            elif m_name == 'DFL':
                if hasattr(m, 'proj') and m.proj is not None:
                    m.proj = m.proj.clone()
                    
        self.model.eval()

        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"  [YOLO] Loaded {model_name} | params={n_params:,} "
              f"| target_cls={self.target_classes}")
        print(f"         conf_thr={conf_threshold} | iou_thr={iou_threshold}")

    @torch.no_grad()
    def count_boxes(self, x_tensor: Tensor,
                    conf_thr: float = None) -> int:
        """
        Count the number of bounding boxes retained after Non-Maximum Suppression (NMS).

        Args:
            x_tensor: Input image tensor of shape (1, C, H, W) with values in [0, 1].
                      Note: This is the input image, not the raw model output.
                      The function performs inference and applies NMS internally.
        Returns:
            The number of bounding boxes corresponding to the target classes after NMS.
        """
        thr = conf_thr or self.conf_threshold
        # Convert the tensor to a NumPy uint8 array for the Ultralytics high-level API
        img_np = (x_tensor.squeeze(0).cpu().numpy()
                  .transpose(1, 2, 0) * 255).astype(np.uint8)

        img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

        results = self._yolo_hl(
            img_np,
            verbose  = False,
            conf     = thr,
            iou      = self.iou_threshold,
            classes  = list(self.target_classes),  
            imgsz    = self.img_size,
        )
        if results and results[0].boxes is not None:
            return len(results[0].boxes)
        return 0

    @torch.no_grad()
    def get_boxes(self, x_tensor: Tensor,
                  conf_thr: float = None) -> List[Dict]:
        """
        Return a list of bounding boxes retained after NMS. 
        Each box is represented as a dictionary: {cls, conf, xyxy: [x1, y1, x2, y2]}.
        """
        thr    = conf_thr or self.conf_threshold
        img_np = (x_tensor.squeeze(0).cpu().numpy()
                  .transpose(1, 2, 0) * 255).astype(np.uint8)
        results = self._yolo_hl(
            img_np, verbose=False, conf=thr,
            iou=self.iou_threshold,
            classes=list(self.target_classes),
            imgsz=self.img_size,
        )
        boxes = []
        if results and results[0].boxes is not None:
            for box in results[0].boxes:
                boxes.append({
                    "cls":  int(box.cls.item()),
                    "conf": float(box.conf.item()),
                    "xyxy": box.xyxy.squeeze().tolist(),
                })
        return boxes

    @torch.no_grad()
    def get_mean_conf(self, x_tensor: Tensor,
                      conf_thr: float = None) -> float:
        """Compute and return the mean confidence score of the detected bounding boxes after NMS."""
        boxes = self.get_boxes(x_tensor, conf_thr)
        if not boxes:
            return 0.0
        return float(np.mean([b["conf"] for b in boxes]))


    def forward_grad(self, x_adv: Tensor):
        """
        Forward pass with gradient tracking enabled.
        
        Args:
            x_adv: Adversarial image tensor of shape (1, C, H, W) with values in [0, 1].
            
        Returns:
            Raw model output tensor. This is strictly used for backpropagation (backward()) 
            and does not perform bounding box counting.
        """
        self.model.eval() 
        out = self.model(x_adv)
        return out

    @torch.no_grad()
    def forward_eval(self, x: Tensor):
        """
        Return the raw tensor output — strictly for internal use within the gradient computation.
        """
        self.model.eval()
        return self.model(x)

print("[OK] FrozenYOLO (FIXED: count_boxes now utilizes NMS) defined.")



[OK] FrozenYOLO (FIXED: count_boxes dùng NMS) defined.


In [12]:
# CELL 8 — GRADIENT COMPUTATION (YOLO-aware L_fg, L_bg, L_conf)
#
# Idea: Use NMS output (no_grad) to identify FG/BG predictions,
# then compute the loss on the raw tensor (with grad) → backpropagate through δ.
#
# FG = predictions matching GT (IoU ≥ 0.5, correct class, conf ≥ τ)
# BG = predictions not matching GT (false positives, conf ≥ τ)
#
# L_conf = mean(p_i for i in FG)                        → minimize to suppress true positives
# L_fg   = mean(p_i * IoU(b_i, b_gt) for i in FG)      → degrade both confidence and localization
# L_bg   = mean(p_j for j in BG)                        → avoid hallucinations
#
# Total = -γ·L_conf - α·L_fg - β·L_bg   (AO-Exp ascent → return gradient)

import torch
from torch import Tensor
import torch.nn.functional as F


def _box_iou_1to1(box_pred: Tensor, box_gt: Tensor) -> Tensor:
    """
    Compute the Intersection over Union (IoU) between one predicted box and one Ground Truth (GT) box.
    Box format: (x1, y1, x2, y2) in pixel scale.
    Returns a scalar tensor (differentiable w.r.t. box_pred if needed,
    but here it is only used as a weight without requiring gradients through the IoU).
    """
    px1, py1, px2, py2 = box_pred
    gx1, gy1, gx2, gy2 = box_gt

    inter_x1 = max(px1, gx1);  inter_y1 = max(py1, gy1)
    inter_x2 = min(px2, gx2);  inter_y2 = min(py2, gy2)
    inter_w = max(inter_x2 - inter_x1, 0)
    inter_h = max(inter_y2 - inter_y1, 0)
    inter   = inter_w * inter_h

    area_p = max(px2 - px1, 0) * max(py2 - py1, 0)
    area_g = max(gx2 - gx1, 0) * max(gy2 - gy1, 0)
    union  = area_p + area_g - inter + 1e-6
    return inter / union


def _decode_raw_boxes(raw_out: Tensor, img_size: int) -> Tensor:
    """
    Decode raw YOLO output into boxes in pixel scale.
    raw_out shape: (N_anchors, 4+nc) — transposed if necessary.
    The first 4 values are (cx, cy, w, h) normalized in [0, 1] after dividing by img_size.
    Returns (N_anchors, 4) in xyxy format in pixel scale.
    
    Note: Raw box coordinates from the YOLO Detect head are in pixel scale (after multiplying by strides),
    we convert them to xyxy format.
    """
    # Raw coordinates: (cx, cy, w, h) in pixel scale (YOLO has already multiplied by strides)
    cx = raw_out[:, 0];  cy = raw_out[:, 1]
    w  = raw_out[:, 2];  h  = raw_out[:, 3]
    x1 = cx - w / 2;    y1 = cy - h / 2
    x2 = cx + w / 2;    y2 = cy + h / 2
    return torch.stack([x1, y1, x2, y2], dim=1)  # (N, 4)


def compute_gradient_wrt_delta(
        frozen_model,
        x_clean:        Tensor,
        delta:          Tensor,
        target_classes: set,
        alpha: float, gamma: float, beta: float,
        conf_threshold: float,
        device: torch.device,
        iou_threshold:  float = 0.5,
) -> Tensor:
    """
    Compute the gradient of L_total with respect to delta.

    L_total = gamma*L_conf + alpha*L_fg - beta*L_bg
    (Positive sign because AO-Exp maximizes the loss, so we aim to INCREASE the loss = suppress detections)

    Strategy:
      - NMS output (no_grad): identify FG/BG anchor indices.
      - Raw output (with_grad): compute the loss on those anchors → backpropagate.
    """
    EPS = 1e-6
    delta_g = delta.detach().clone().requires_grad_(True)
    x_adv   = torch.clamp(
        x_clean.unsqueeze(0) + delta_g.unsqueeze(0), 0.0, 1.0)

    H, W    = x_adv.shape[2], x_adv.shape[3]
    img_sz  = frozen_model.img_size

    # ── STEP 1: Extract GT boxes from the clean image (NMS, no_grad) ─────────
    # gt_boxes = list of dicts {"xyxy": [x1, y1, x2, y2], "cls": int}
    with torch.no_grad():
        gt_boxes = frozen_model.get_boxes(
            x_clean.unsqueeze(0), conf_threshold)
        # Filter for target classes only
        gt_boxes = [b for b in gt_boxes if b["cls"] in target_classes]

    # ── STEP 2: Forward pass of x_adv through the raw model (with grad) ──────
    raw_out = frozen_model.forward_grad(x_adv)
    r = raw_out[0] if isinstance(raw_out, (list, tuple)) else raw_out
    # r shape: (B, N_anchors, 4+nc) or (B, 4+nc, N_anchors)
    if r.dim() == 3 and r.shape[1] < r.shape[2]:
        r = r.transpose(1, 2)   # → (B, N_anchors, 4+nc)
    r = r[0]                     # (N_anchors, 4+nc)

    nc          = r.shape[1] - 4
    raw_coords  = r[:, :4]                          # (N, 4) box coords
    cls_logits  = r[:, 4:]                          # (N, nc) logits
    cls_scores  = cls_logits.sigmoid()              # (N, nc)

    # ── STEP 3: Compute scores for target classes ────────────────────────────
    tc_list = sorted(target_classes)
    if len(tc_list) == 0 or max(tc_list) >= nc:
        # Fallback: use the maximum score
        adv_conf = cls_scores.max(dim=-1).values    # (N,)
    else:
        # Confidence = maximum score across target classes
        tc_idx   = [tc for tc in tc_list if tc < nc]
        adv_conf = cls_scores[:, tc_idx].max(dim=-1).values  # (N,)

    # ── STEP 4: Decode boxes (no_grad to obtain pixel coordinates) ───────────
    with torch.no_grad():
        pred_boxes_xyxy = _decode_raw_boxes(raw_coords.detach(), img_sz)
        # (N_anchors, 4) xyxy in pixel scale

    # ── STEP 5: Identify FG / BG anchors based on NMS predictions ────────────
    # Use NMS output of the adversarial image to classify anchors (no_grad)
    with torch.no_grad():
        adv_nms_boxes = frozen_model.get_boxes(x_adv.detach(), conf_threshold)
        adv_nms_boxes = [b for b in adv_nms_boxes if b["cls"] in target_classes]

    # Map NMS boxes → closest raw anchor (by center distance)
    # FG anchors: anchors with IoU with GT ≥ iou_threshold AND correct class
    # BG anchors: anchors with conf ≥ τ but IoU with all GTs < iou_threshold

    fg_indices  = []
    bg_indices  = []
    iou_weights = []   # IoU(b_pred, b_gt) for each FG anchor

    # Confidence threshold to select "active" anchors (avoid 8400 zeros)
    # Use detached confidence to avoid creating duplicate computation graphs
    with torch.no_grad():
        conf_mask = adv_conf.detach() >= conf_threshold
        active_idx = conf_mask.nonzero(as_tuple=True)[0].tolist()

    if len(active_idx) == 0:
        # If no predictions are found → use the top-50 anchors
        active_idx = adv_conf.detach().topk(min(50, adv_conf.shape[0])).indices.tolist()

    for idx in active_idx:
        pred_box = pred_boxes_xyxy[idx].tolist()  # [x1, y1, x2, y2]
        best_iou = 0.0
        best_gt  = None
        for gt in gt_boxes:
            iou_val = _box_iou_1to1(pred_box, gt["xyxy"])
            if iou_val > best_iou:
                best_iou = iou_val
                best_gt  = gt

        if best_iou >= iou_threshold and best_gt is not None:
            fg_indices.append(idx)
            iou_weights.append(best_iou)
        else:
            bg_indices.append(idx)

    # ── STEP 6: Compute loss components (WITH grad) ──────────────────────────

    # L_conf: mean confidence of FG anchors
    if len(fg_indices) > 0:
        fg_idx_t   = torch.tensor(fg_indices, device=device)
        fg_confs   = adv_conf[fg_idx_t]                      # (n_fg,) with grad
        L_conf     = fg_confs.mean()

        # L_fg: confidence weighted by IoU
        iou_w_t    = torch.tensor(iou_weights, dtype=torch.float32, device=device)
        L_fg       = (fg_confs * iou_w_t).mean()
    else:
        # Fallback: no FG anchors → use global top-K
        top_k      = min(50, adv_conf.shape[0])
        topk_vals  = adv_conf.topk(top_k).values
        L_conf     = topk_vals.mean()
        L_fg       = L_conf

    # L_bg: mean confidence of BG anchors (false positives)
    if len(bg_indices) > 0:
        bg_idx_t = torch.tensor(bg_indices, device=device)
        # Select the top-50 BG anchors (avoid noise from too many zero-confidence anchors)
        bg_confs = adv_conf[bg_idx_t]
        if bg_confs.shape[0] > 50:
            bg_confs = bg_confs.topk(50).values
        L_bg = bg_confs.mean()
    else:
        L_bg = torch.tensor(0.0, device=device)

    # ── STEP 7: Total loss ───────────────────────────────────────────────────
    # Since AO-Exp maximizes the loss → we aim to suppress FG (increase L_conf, L_fg)
    # and suppress FP hallucinations (decrease L_bg)
    # total_loss = gamma*L_conf + alpha*L_fg - beta*L_bg
    total_loss = -gamma * L_conf - alpha * L_fg - beta * L_bg

    # ── STEP 8: Backward pass ────────────────────────────────────────────────
    if total_loss.requires_grad:
        total_loss.backward()
        grad = delta_g.grad.clone() if delta_g.grad is not None \
               else torch.zeros_like(delta)
    else:
        grad = torch.zeros_like(delta)

    return grad

def compute_batch_gradient(
    frozen_model,       # FrozenYOLO
    batch_frames: list, # List[Tensor]
    delta:          Tensor,
    target_classes: set,
    alpha: float, gamma: float, beta: float,
    conf_threshold: float,
    device: torch.device,
) -> Tensor:
    """
    Compute the average gradient over a batch of frames.
    Matches exactly Eq. (1) in the paper: ∇G(δ) = (1/B) * Σ_{b=1}^B ∇L_total(...)
    """
    grad_accum = torch.zeros_like(delta)
    B = len(batch_frames)
    for x_clean in batch_frames:
        g = compute_gradient_wrt_delta(
            frozen_model=frozen_model, 
            x_clean=x_clean, 
            delta=delta,
            target_classes=target_classes,
            alpha=alpha, gamma=gamma, beta=beta,
            conf_threshold=conf_threshold, 
            device=device,
        )
        grad_accum = grad_accum + g / B
    return grad_accum

In [13]:
# CELL 9 — AO-Exp OPTIMIZER
# Implements Algorithm 1: Adaptive Optimistic Exponentiated Gradient
# Paper equations: (2) optimistic update, (3) Lambert W, (4) perturbation reconstruction

class AOExpOptimizer:
    """
    Adaptive Optimistic Exponentiated Gradient optimizer cho UAP.
    
    State per channel c:
      U_c  ∈ R^{H×H}  : left singular vectors (orthogonal basis)
      V_c  ∈ R^{W×W}  : right singular vectors (orthogonal basis)
      z_c  ∈ R^r_≥0   : singular values
      η_c  : adaptive step size

    δ^c_t = (2/t(t+1)) Σ_{s=1}^t  s · U_c diag(z^c_{s,1:k}) V_c^T   [eq.4]
    """

    def __init__(self,
                 H: int, W: int, C: int,
                 lambda1:   float = 0.1,
                 lambda2:   float = 0.0002,
                 eta_init:  float = 1.0,
                 k:         int   = 10,
                 device: torch.device = torch.device("cpu")):
        self.H, self.W, self.C = H, W, C
        self.lambda1  = lambda1
        self.lambda2  = lambda2
        self.k        = k
        self.device   = device
        self.r        = min(H, W)    # max rank
        self.t        = 0            # iteration counter

        # ── Per-channel state ─────────────────────────────────────────────
        self.U         = [torch.eye(H, device=device)           for _ in range(C)]
        self.V         = [torch.eye(W, device=device)           for _ in range(C)]
        self.z         = [torch.zeros(self.r, device=device)    for _ in range(C)]
        self.theta     = [torch.zeros(self.r, device=device)    for _ in range(C)]
        self.eta       = [float(eta_init)]                     * C
        self.prev_grad = [None]                                * C
        # Weighted accumulator Σ s·z^c_s  (for eq.4)
        self.z_accum   = [torch.zeros(self.r, device=device)   for _ in range(C)]

        print(f"  [AO-Exp] Init: H={H} W={W} C={C} r={self.r} k={k} "
              f"λ₁={lambda1} λ₂={lambda2} η₀={eta_init}")

    # ── Lambert W₀ ───────────────────────────────────────────────────────
    @staticmethod
    def _lambert_w0(x: np.ndarray) -> np.ndarray:
        """Principal branch W₀(x) của Lambert W: W₀(x)·exp(W₀(x)) = x."""
        return np.real(lambertw(x, k=0))

    # ── Eq.(3): singular value update ────────────────────────────────────
    def _z_update(self, theta_c: Tensor, eta_c: float) -> Tensor:
        """
        z_i = (η/λ₂) · W₀( (λ₂/η) · exp((λ₂ + max{θ_i−λ₁, 0}) / η) ) − 1
        """
        lam1, lam2 = self.lambda1, self.lambda2
        θ = theta_c.cpu().numpy()

        exp_arg = (lam2 + np.maximum(θ - lam1, 0.0)) / eta_c
        exp_arg = np.clip(exp_arg, -50.0, 50.0)           # stability

        lambert_arg = (lam2 / eta_c) * np.exp(exp_arg)
        w0          = self._lambert_w0(lambert_arg)

        z_new = (eta_c / lam2) * w0 - 1.0
        z_new = np.maximum(z_new, 0.0)                    # z ≥ 0

        return torch.tensor(z_new, dtype=torch.float32, device=self.device)

    # ── Main step ─────────────────────────────────────────────────────────
    def step(self, grad_G: Tensor) -> Tensor:
        """
        Một bước AO-Exp update.
        grad_G: (C, H, W) — gradient trung bình trên batch.
        Trả về δ mới shape (C, H, W).
        """
        self.t += 1
        t = self.t
        delta_channels = []

        for c in range(self.C):
            g_curr = grad_G[c]                            # (H, W)
            g_prev = self.prev_grad[c] if self.prev_grad[c] is not None \
                     else torch.zeros_like(g_curr)

            # ── (1) Adaptive step size  [eq.2, dòng 1] ───────────────────
            diff_inf   = (g_curr - g_prev).abs().max().item()
            eta_c      = self.eta[c] + float(t)**2 * diff_inf**2
            self.eta[c] = eta_c

            # ── (2) Optimistic update  [eq.2] ────────────────────────────
            # z̄^c_{t,i} = log(z^c_{t,i} + 1)
            z_bar = torch.log(self.z[c] + 1.0)           # (r,)

            r = self.r
            # Low-rank matrix trong log-space
            LR = eta_c * (self.U[c][:, :r]
                          @ torch.diag(z_bar)
                          @ self.V[c][:r, :])             # (H,W)

            # M = η·U·diag(z̄)·Vᵀ + (2t+1)·g_curr − t·g_prev
            M = LR + (2*t + 1) * g_curr - t * g_prev     # (H,W)

            # SVD(M) → new orthogonal bases + intermediate singular values θ
            try:
                U_new, theta_vals, Vh_new = torch.linalg.svd(
                    M, full_matrices=True)
            except Exception:
                # fallback: keep previous bases
                U_new, theta_vals, Vh_new = \
                    self.U[c], self.theta[c][:min(r, self.H)], self.V[c]

            r_svd = theta_vals.shape[0]
            theta_c = torch.zeros(r, device=self.device)
            theta_c[:min(r_svd, r)] = theta_vals[:min(r_svd, r)]

            self.U[c]     = U_new
            self.V[c]     = Vh_new
            self.theta[c] = theta_c

            # ── (3) Singular value update via Lambert W  [eq.3] ──────────
            z_new       = self._z_update(theta_c, eta_c)
            self.z[c]   = z_new

            # ── Top-k truncation (promote low-rank) ───────────────────────
            z_topk = torch.zeros_like(z_new)
            k_act  = min(self.k, r)
            topk_v, topk_i = torch.topk(z_new, k_act)
            z_topk[topk_i] = topk_v

            # ── (4) Weighted average reconstruction  [eq.4] ───────────────
            # z_accum += t · z_topk
            self.z_accum[c] = self.z_accum[c] + t * z_topk
            weight     = 2.0 / (t * (t + 1))
            z_weighted = weight * self.z_accum[c]         # (r,)

            delta_c = (U_new[:, :r]
                       @ torch.diag(z_weighted)
                       @ Vh_new[:r, :])                    # (H,W)
            delta_channels.append(delta_c)

            self.prev_grad[c] = g_curr.clone()

        delta = torch.stack(delta_channels, dim=0)        # (C, H, W)
        return delta

    @property
    def nuclear_norm(self) -> float:
        """‖δ‖_* = Σ_c Σ_i σ_i^c (sum of all singular values)."""
        return sum(self.z[c].sum().item() for c in range(self.C))

print("[OK] AOExpOptimizer defined.")

[OK] AOExpOptimizer defined.


In [14]:
# CELL 10 — BASELINE: Low-Rank PGD (LoRa-PGD)
# Section 5.1 — paper baseline

class LoRaPGDAttack:
    """
    Low-Rank PGD: δ = U ⊗ V
      U ∈ R^{H×r×C},  V ∈ R^{r×W×C},  r ≤ min{H,W}
    Nuclear norm constraint: ‖U⊗V‖_* ≤ nuc_budget
    """

    def __init__(self, H: int, W: int, C: int,
                 rank_ratio:  float = 0.10,
                 nuc_budget:  float = 60.0,
                 lr:          float = 0.05,
                 device: torch.device = torch.device("cpu")):
        self.H, self.W, self.C = H, W, C
        self.r          = max(1, int(min(H, W) * rank_ratio))
        self.nuc_budget = nuc_budget
        self.lr         = lr
        self.device     = device

        # Factor matrices (learnable, NOT nn.Parameter — we update manually)
        self.U_mat = torch.randn(H, self.r, C, device=device) * 0.01
        self.V_mat = torch.randn(self.r, W, C, device=device) * 0.01
        print(f"  [LoRa-PGD] r={self.r} | nuc_budget={nuc_budget} | lr={lr}")

    def get_perturbation(self) -> Tensor:
        """δ^c_{ij} = Σ_k U_{ik,c}·V_{kj,c}  →  shape (C,H,W)."""
        # einsum: hrc, rwc -> hwc
        delta = torch.einsum("hrc,rwc->hwc", self.U_mat, self.V_mat)
        return delta.permute(2, 0, 1)   # (C, H, W)

    def step(self, grad_G: Tensor):
        """
        Gradient ascent step on U, V.
        grad_G: (C, H, W)
        """
        grad_perm = grad_G.permute(1, 2, 0)   # (H, W, C)
        grad_U = torch.einsum("hwc,rwc->hrc", grad_perm, self.V_mat)
        grad_V = torch.einsum("hrc,hwc->rwc", self.U_mat, grad_perm)

        norm_U = grad_U.norm() + 1e-8
        norm_V = grad_V.norm() + 1e-8

        self.U_mat = self.U_mat + self.lr * grad_U / norm_U
        self.V_mat = self.V_mat + self.lr * grad_V / norm_V

        self._project_nuclear()

    def _project_nuclear(self):
        """Project δ vào nuclear norm ball ‖δ‖_* ≤ nuc_budget."""
        delta = self.get_perturbation()    # (C, H, W)
        for c in range(self.C):
            d_c = delta[c]
            try:
                _, s_, _ = torch.linalg.svd(d_c, full_matrices=False)
            except Exception:
                continue
            nuc = s_.sum().item()
            if nuc > self.nuc_budget:
                scale = (self.nuc_budget / nuc) ** 0.5
                self.U_mat[:, :, c] *= scale
                self.V_mat[:, :, c] *= scale

print("[OK] LoRaPGDAttack defined.")

[OK] LoRaPGDAttack defined.


In [15]:
# CELL 11A — HELPER & EVALUATION METRICS 

import numpy as np

def compute_iou(box1, box2):
    """Compute the Intersection over Union (IoU) between two bounding boxes in [x1, y1, x2, y2] format."""
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    inter_x1 = max(x1_1, x1_2)
    inter_y1 = max(y1_1, y1_2)
    inter_x2 = min(x2_1, x2_2)
    inter_y2 = min(y2_1, y2_2)
    if inter_x2 < inter_x1 or inter_y2 < inter_y1:
        return 0.0
    inter_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
    area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
    area2 = (x2_2 - x1_2) * (y2_2 - y1_2)
    return inter_area / (area1 + area2 - inter_area + 1e-6)

def match_boxes_with_iou(clean_boxes, adv_boxes, iou_thresh=0.5, match_class=True):
    """
    Match clean and adversarial bounding boxes using Greedy IoU matching (COCO standard).
    Returns: matched pairs (True Positives), fp_indices (False Positives), fn_indices (False Negatives).
    """
    n_c, n_a = len(clean_boxes), len(adv_boxes)
    iou_mat = np.zeros((n_c, n_a))
    for i in range(n_c):
        for j in range(n_a):
            iou_val = compute_iou(clean_boxes[i]['xyxy'], adv_boxes[j]['xyxy'])
            if match_class and clean_boxes[i]['cls'] != adv_boxes[j]['cls']:
                iou_val = 0.0  # Different class → considered a mismatch
            iou_mat[i, j] = iou_val

    matched_c, matched_a = set(), set()
    pairs = []
    
    # Prioritize matching adversarial boxes with higher confidence scores first
    for j in sorted(range(n_a), key=lambda k: adv_boxes[k]['conf'], reverse=True):
        if j in matched_a: continue
        best_i, best_iou = -1, 0.0
        for i in range(n_c):
            if i in matched_c: continue
            if iou_mat[i, j] > best_iou:
                best_iou, best_i = iou_mat[i, j], i
        if best_i != -1 and best_iou >= iou_thresh:
            matched_c.add(best_i)
            matched_a.add(j)
            pairs.append((best_i, j, best_iou))

    fn_idx = [i for i in range(n_c) if i not in matched_c]
    fp_idx = [j for j in range(n_a) if j not in matched_a]
    return pairs, fp_idx, fn_idx

from scipy.optimize import linear_sum_assignment
from torchvision.ops import box_iou

@torch.no_grad()
def compute_metrics_full(
        frozen_model: FrozenYOLO,
        frames: List[Tensor],
        delta: Tensor,
        conf_threshold: float = 0.25,
        iou_match_thr: float = 0.5
) -> Dict:
    """
    Compute evaluation metrics compatible with the AO-Exp paper (Section 5.2), utilizing Hungarian Matching.
    Returns a comprehensive set of metrics for evaluation: Suppression (advBR), Localization (IoU_acc), 
    Confidence Degradation (conf_drop_matched), and Stealthiness (MAP, nuc_norm).
    """
    frozen_model.model.eval()
    total_nc, total_na = 0, 0
    total_iou_sum = 0.0
    total_matched = 0
    sum_conf_clean_matched, sum_conf_adv_matched = 0.0, 0.0

    for x_clean in tqdm(frames, desc="  Eval", leave=False):
        x_adv = torch.clamp(x_clean + delta, 0.0, 1.0)
        
        clean_boxes = frozen_model.get_boxes(x_clean.unsqueeze(0), conf_threshold)
        adv_boxes   = frozen_model.get_boxes(x_adv.unsqueeze(0), conf_threshold)
        
        nc, na = len(clean_boxes), len(adv_boxes)
        total_nc += nc
        total_na += na
        
        if nc > 0 and na > 0:
            c_boxes = torch.tensor([b['xyxy'] for b in clean_boxes])
            a_boxes = torch.tensor([b['xyxy'] for b in adv_boxes])
            
            # IoU matrix
            iou_mat = box_iou(c_boxes, a_boxes)
            # Global optimal Hungarian Matching
            row_ind, col_ind = linear_sum_assignment(-iou_mat.cpu().numpy())
            
            for r, c in zip(row_ind, col_ind):
                if iou_mat[r, c] >= iou_match_thr:
                    total_iou_sum += iou_mat[r, c].item()
                    total_matched += 1
                    # Track the confidence scores of the matched pairs
                    sum_conf_clean_matched += clean_boxes[r]['conf']
                    sum_conf_adv_matched   += adv_boxes[c]['conf']

    T = len(frames)
    
    # === Primary metrics (retaining original names and formulas from the paper) ===
    IoU_acc  = total_iou_sum / max(T, 1)  # Accumulated Matched IoU per Frame
    advBR    = total_na / max(total_nc, 1)
    MAP      = delta.abs().mean().item()
    nuc_norm = sum(torch.linalg.svdvals(delta[c]).sum().item() for c in range(delta.shape[0]))
    
    # === Auxiliary metrics (academic standard, used for in-depth analysis and defense evaluation) ===
    mIoU_matched = total_iou_sum / max(total_matched, 1)  # Mean IoU per matched pair
    conf_drop_matched = (1.0 - sum_conf_adv_matched / max(sum_conf_clean_matched, 1e-8)) * 100
    
    return {
        # Paper-compatible (used in Cells 12, 15, 17, and 47)
        "IoU_acc":  round(IoU_acc, 6),
        "advBR":    round(advBR, 6),
        "MAP":      round(MAP, 6),
        "nuc_norm": round(nuc_norm, 4),
        
        # Rigorous auxiliary metrics (used for in-depth analysis tables and defense evaluation)
        "mIoU_matched": round(mIoU_matched, 6),
        "conf_drop_matched": round(conf_drop_matched, 2),
        "n_clean":  total_nc,
        "n_adv":    total_na,
        "n_matched": total_matched
    }

print("[OK] compute_metrics_full (Rigorous v2 + Confidence Drop) defined.")

[OK] compute_metrics_full (Rigorous v2 + Confidence Drop) defined.


In [16]:
# CELL 12 — UAP TRAINER 

class UAPTrainer:
    def __init__(self, frozen_model: FrozenYOLO,
                 train_frames: List[Tensor], eval_frames: List[Tensor], 
                 img_size: int, target_classes: set, lambda1: float, lambda2: float,
                 eta_init: float, n_iter: int, k_rank: int, batch_frames: int,
                 alpha: float, gamma: float, beta: float, conf_threshold: float,
                 epsilon: float, device: torch.device, video_name: str, class_mode: str):
        self.model = frozen_model
        self.train_frames = train_frames # ← Used for the training phase
        self.eval_frames = eval_frames   # ← Used for the evaluation phase
        self.img_size = img_size
        self.target_classes = target_classes
        self.n_iter = n_iter
        self.batch_frames = batch_frames
        self.alpha, self.gamma, self.beta = alpha, gamma, beta
        self.conf_threshold = conf_threshold
        self.epsilon = epsilon
        self.device = device
        self.video_name = video_name
        self.class_mode = class_mode
        
        C, H, W = 3, img_size, img_size
        self.optimizer = AOExpOptimizer(H=H, W=W, C=C, lambda1=lambda1, lambda2=lambda2,
                                        eta_init=eta_init, k=k_rank, device=device)
        self.delta = torch.zeros(C, H, W, device=device)
        self.log = {"iter":[], "advBR":[], "IoU_acc":[], "nuc_norm":[], "map":[], "n_clean":[], "n_adv":[]}

    def _sample_batch_temporal(self) -> List[Tensor]:
        """Sample a batch from the training set to compute the gradient."""
        N = len(self.train_frames)
        B = min(self.batch_frames, N)
        bucket_size = N / B
        selected = []
        for b in range(B):
            lo = int(b * bucket_size)
            hi = int((b + 1) * bucket_size)
            hi = min(hi, N)
            if lo >= hi: lo = max(0, hi - 1)
            idx = random.randint(lo, hi - 1)
            selected.append(self.train_frames[idx])
        return selected

    @torch.no_grad()
    def _quick_eval(self, n_eval: int = 20) -> Dict:
        """
        Evaluate on the evaluation/test set using the NMS count, which represents the actual number of detected bounding boxes.
        """
        N = len(self.eval_frames)
        eval_indices = VideoFrameDataset._uniform_indices(N, min(n_eval, N))
        subset = [self.eval_frames[i] for i in eval_indices]

        total_clean, total_adv, iou_sum = 0, 0, 0.0
        self.model.model.eval()

        for x in subset:
            x_adv = torch.clamp(x + self.delta, 0.0, 1.0)
            # Use the updated count_boxes method (takes an image tensor and returns the NMS count)
            nc = self.model.count_boxes(x.unsqueeze(0),    self.conf_threshold)
            na = self.model.count_boxes(x_adv.unsqueeze(0), self.conf_threshold)
            total_clean += nc
            total_adv   += na
            iou_sum     += na / max(nc, 1)

        T = len(subset)
        return {
            "advBR":    total_adv / max(total_clean, 1),
            "IoU_acc":  iou_sum / T,
            "nuc_norm": self.optimizer.nuclear_norm,
            "map":      self.delta.abs().mean().item(),
            "n_clean":  total_clean,
            "n_adv":    total_adv,
        }
        
    def train(self) -> Tensor:
        print(f"\n{'='*62}")
        print(f"  AO-Exp Attack | video={self.video_name} | class={self.class_mode}")
        print(f"  Train frames:{len(self.train_frames)} | Eval frames:{len(self.eval_frames)}")
        print(f"{'='*62}")
    
        best_delta = self.delta.clone()
        best_adv_br = 1.0
        eval_every = max(1, self.n_iter // 10)
        pbar = tqdm(range(1, self.n_iter + 1), desc="AO-Exp ", unit="it")
    
        # 🔹 Prepare the clipping threshold (convert from the 0-255 scale to the 0-1 scale)
        #eps_norm = self.epsilon / 255.0 if self.epsilon > 0 else float('inf')
    
        for t in pbar:
            batch = self._sample_batch_temporal()
            grad_G = compute_batch_gradient(
                frozen_model=self.model, batch_frames=batch, delta=self.delta,
                target_classes=self.target_classes, alpha=self.alpha, gamma=self.gamma,
                beta=self.beta, conf_threshold=self.conf_threshold, device=self.device
            )
            self.delta = self.optimizer.step(grad_G)
    
            # 🔹 Apply the epsilon constraint (L-infinity norm constraint)
            # Ensure that the perturbation at each pixel does not exceed +/- epsilon/255
            #if eps_norm != float('inf'):
                #self.delta = torch.clamp(self.delta, -eps_norm, eps_norm)
    
            if t % eval_every == 0 or t == self.n_iter:
                m = self._quick_eval(n_eval=20)
                self.log["iter"].append(t)
                for k in ["advBR", "IoU_acc", "nuc_norm", "map", "n_clean", "n_adv"]:
                    self.log[k].append(m[k])
                if m["advBR"] < best_adv_br:
                    best_adv_br = m["advBR"]
                    best_delta = self.delta.clone()
                pbar.set_postfix({"advBR": f"{m['advBR']:.3f}", "‖δ‖_*": f"{m['nuc_norm']:.1f}",
                                  "MAP": f"{m['map']:.4f}", "det": f"{m['n_adv']}/{m['n_clean']}"})
    
        print(f"\n[TRAIN] Done. Best advBR={best_adv_br:.4f}")
        self.delta = best_delta
        return best_delta

print("[OK] UAPTrainer (stratified temporal sampling) defined.")

[OK] UAPTrainer (stratified temporal sampling) defined.


In [17]:
# CELL 13 — VISUALIZATION FUNCTIONS

def visualize_perturbation(delta:      Tensor,
                            video_name: str,
                            class_mode: str,
                            save_dir:   Path,
                            sample_frame: Tensor = None):
    """
    Vẽ 3 panel:
      1. Perturbation magnitude heatmap
      2. Singular value spectrum per channel
      3. Clean vs. Adversarial frame (nếu có frame mẫu)
    """
    delta_np  = delta.cpu().numpy()               # (C, H, W)
    delta_mag = np.abs(delta_np).max(axis=0)      # (H, W) — max across channels

    n_panels = 3 if sample_frame is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(6*n_panels, 5))
    fig.suptitle(f"UAP  |  video={video_name}  |  class={class_mode}",
                 fontsize=12, fontweight="bold")

    # ── Panel 1: magnitude heatmap ────────────────────────────────────────
    ax = axes[0]
    im = ax.imshow(delta_mag, cmap="hot", vmin=0,
                   vmax=max(delta_mag.max(), 1e-6))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title("Perturbation magnitude\n(max over RGB channels)")
    ax.axis("off")

    # ── Panel 2: singular value spectrum ─────────────────────────────────
    ax = axes[1]
    colors = ["#e74c3c", "#2ecc71", "#3498db"]
    labels = ["R", "G", "B"]
    for c, (col, lbl) in enumerate(zip(colors, labels)):
        sv = torch.linalg.svdvals(delta[c]).cpu().numpy()
        sv_pos = sv[sv > 1e-12]
        if sv_pos.size > 0:
            ax.semilogy(sv_pos, color=col, alpha=0.8,
                        linewidth=1.5, label=lbl)
    ax.set_title("Singular value spectrum\n(log scale)")
    ax.set_xlabel("Rank index")
    ax.set_ylabel("Singular value")
    ax.legend(loc="upper right")
    ax.grid(True, which="both", alpha=0.3, linestyle="--")

    # ── Panel 3: clean vs. adversarial ───────────────────────────────────
    if sample_frame is not None:
        ax = axes[2]
        x_adv = torch.clamp(sample_frame + delta, 0.0, 1.0)
        # Stack side-by-side
        side = torch.cat([sample_frame.cpu(),
                           x_adv.cpu()], dim=2)           # (C, H, 2W)
        side_np = side.permute(1,2,0).numpy()
        side_np = np.clip(side_np, 0, 1)
        ax.imshow(side_np)
        ax.set_title("Clean (left)  vs.  Adversarial (right)")
        ax.axis("off")
        # Separator line
        W_ = sample_frame.shape[-1]
        ax.axvline(x=W_, color="yellow", linewidth=2, linestyle="--")

    plt.tight_layout()
    save_path = save_dir / f"perturbation_{video_name}_{class_mode}.png"
    plt.savefig(str(save_path), dpi=130, bbox_inches="tight")
    plt.show()
    print(f"  [VIZ] Saved → {save_path}")


def plot_training_curves(log: Dict, video_name: str,
                          class_mode: str, save_dir: Path):
    """Plot advBR, nuclear norm, MAP theo số iteration."""
    if not log.get("iter"):
        print("  [VIZ] No log to plot yet.")
        return

    iters = log["iter"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Training curves  |  {video_name}  |  {class_mode}",
                 fontsize=11)

    # advBR
    axes[0].plot(iters, log["advBR"], "b-o", ms=4, lw=1.5)
    axes[0].axhline(0, color="green", lw=1, linestyle="--", label="ideal=0")
    axes[0].set_title("Adversarial Box Ratio (advBR ↓)")
    axes[0].set_xlabel("Iteration")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # nuclear norm
    axes[1].plot(iters, log["nuc_norm"], "r-o", ms=4, lw=1.5)
    axes[1].set_title("Nuclear Norm ‖δ‖_* (↓)")
    axes[1].set_xlabel("Iteration")
    axes[1].grid(alpha=0.3)

    # MAP
    axes[2].plot(iters, log["map"], "g-o", ms=4, lw=1.5)
    axes[2].set_title("Mean Absolute Perturbation MAP (↓)")
    axes[2].set_xlabel("Iteration")
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    save_path = save_dir / f"curves_{video_name}_{class_mode}.png"
    plt.savefig(str(save_path), dpi=110, bbox_inches="tight")
    plt.show()
    print(f"  [VIZ] Saved → {save_path}")

print("[OK] Visualization functions defined.")

[OK] Visualization functions defined.


In [18]:
# CELL 14 — SAVE / LOAD PERTURBATION

def save_perturbation(delta:       Tensor,
                       metrics:     Dict,
                       video_name:  str,
                       class_mode:  str,
                       hyperparams: Dict,
                       save_dir:    Path) -> Path:
    
    fname     = f"uap_{video_name}_{class_mode}.pt"
    save_path = save_dir / fname
    payload   = {
        "delta":       delta.cpu(),
        "video_name":  video_name,
        "class_mode":  class_mode,
        "metrics":     metrics,
        "hyperparams": hyperparams,
        "timestamp":   time.strftime("%Y-%m-%d %H:%M:%S"),
        "shape":       tuple(delta.shape),
    }
    torch.save(payload, str(save_path))

    print(f"\n  [SAVE] ✓  {save_path}")
    print(f"         advBR={metrics.get('advBR','?')}  "
          f"‖δ‖_*={metrics.get('nuc_norm','?')}  "
          f"MAP={metrics.get('MAP','?')}")
    return save_path


def load_perturbation(save_dir:   Path,
                       video_name: str,
                       class_mode: str) -> Dict:
    fpath = save_dir / f"uap_{video_name}_{class_mode}.pt"
    if not fpath.exists():
        raise FileNotFoundError(f"Not found: {fpath}")
    data = torch.load(str(fpath), map_location="cpu",
                       weights_only=False)
    print(f"  [LOAD] ✓  {fpath}")
    print(f"         video={data['video_name']}  "
          f"class={data['class_mode']}  "
          f"shape={data['shape']}")
    return data


def list_saved_perturbations(save_dir: Path):
    files = sorted(save_dir.glob("uap_*.pt"))
    if not files:
        print("  [INFO] No perturbations saved yet.")
        return
    print(f"\n  Saved perturbations ({len(files)}):")
    for f in files:
        data = torch.load(str(f), map_location="cpu", weights_only=False)
        m    = data.get("metrics", {})
        print(f"    {f.name:<45}  "
              f"advBR={m.get('advBR','?')}  "
              f"‖δ‖_*={m.get('nuc_norm','?')}  "
              f"@ {data.get('timestamp','?')}")

print("[OK] Save/Load functions defined.")

[OK] Save/Load functions defined.


In [19]:
# CELL 15 — FULL PIPELINE (FIXED: Train/Test Separation)
# Auto-select max_frames theo VIDEO_MAX_FRAMES[video_name]
# ✅ Train trên train.txt | Monitor trên val.txt | Báo cáo trên test.txt

def run_attack_pipeline(
    video_name: str, class_mode: str, dataset_root: str,
    dataset_dir: Path, perturb_dir: Path, viz_dir: Path,
    lambda1: float = LAMBDA1, lambda2: float = LAMBDA2,
    epsilon: float = EPSILON, lr_init: float = LR_INIT,
    n_iter: int = N_ITER, k_rank: int = K_RANK,
    tau: float = TAU, alpha: float = ALPHA,
    gamma: float = GAMMA, beta: float = BETA,
    batch_frames: int = BATCH_FRAMES,
    max_frames: int = None, sample_mode: str = "uniform",
    img_size: int = IMG_SIZE, yolo_model: str = YOLO_MODEL,
    device: torch.device = DEVICE, skip_if_exists: bool = True,
) -> Dict:
    
    pert_path = perturb_dir / f"uap_{video_name}_{class_mode}.pt"
    if skip_if_exists and pert_path.exists():
        print(f"[SKIP] Exists: {pert_path.name}")
        return load_perturbation(perturb_dir, video_name, class_mode)["metrics"]

    target_classes = CLASS_MODE_TO_YOLO[class_mode]
    if max_frames is None:
        max_frames = VIDEO_MAX_FRAMES.get(video_name, 300)

    total_known = VIDEO_FRAME_COUNTS.get(video_name, "?")
    print(f"\n[PIPELINE] {video_name} | {class_mode}")
    print(f"  Total frames: {total_known} | max_frames: {max_frames}")

    def get_paths_from_txt(txt_name: str, vname: str) -> List[str]:
        txt_path = dataset_dir / txt_name
        paths = []
        if txt_path.exists():
            with open(txt_path, 'r') as f:
                for line in f:
                    p = line.strip()
                    if p and vname in p:
                        paths.append(p)
        return paths

    train_paths = get_paths_from_txt("train.txt", video_name)
    val_paths   = get_paths_from_txt("val.txt",   video_name)
    test_paths  = get_paths_from_txt("test.txt",  video_name)

    if not train_paths or not val_paths or not test_paths:
        raise ValueError(
            f"Thiếu dữ liệu split cho {video_name}:\n"
            f"  train:{len(train_paths)} | val:{len(val_paths)} | test:{len(test_paths)}"
        )

    # 1️⃣ TRAIN frames → compute gradient
    train_frames = load_frames_from_path_list(
        path_list=train_paths, img_size=img_size, max_frames=max_frames,
        device=device, sample_mode=sample_mode
    )
    print(f"  ✅ TRAIN: {len(train_frames)} frames (gradient update)")

    # 2️⃣ VAL frames → monitoring & pick best_delta
    val_frames = load_frames_from_path_list(
        path_list=val_paths, img_size=img_size,
        max_frames=min(max_frames, 100),
        device=device, sample_mode="uniform"
    )
    print(f"  ✅ VAL: {len(val_frames)} frames (pick best_delta)")

    # 3️⃣ TEST frames 
    test_frames = load_frames_from_path_list(
        path_list=test_paths, img_size=img_size,
        max_frames=min(max_frames, 100),
        device=device, sample_mode="uniform"
    )
    print(f"  ✅ TEST: {len(test_frames)} frames (final report ONLY)")

    # ── Init frozen model ─────────────────────────────────────────────
    frozen_model = FrozenYOLO(
        model_name=yolo_model, device=device, img_size=img_size,
        target_classes=target_classes, conf_threshold=tau,
    )

    hyperparams = dict(
        lambda1=lambda1, lambda2=lambda2, epsilon=epsilon,
        lr_init=lr_init, n_iter=n_iter, k_rank=k_rank, tau=tau,
        alpha=alpha, gamma=gamma, beta=beta,
        batch_frames=batch_frames, max_frames=max_frames,
        sample_mode=sample_mode, img_size=img_size,
        total_video_frames=total_known,
    )

    trainer = UAPTrainer(
        frozen_model=frozen_model,
        train_frames=train_frames,  # ← Gradient flow
        eval_frames=val_frames,     # ← Monitor & pick best (KHÔNG phải test!)
        img_size=img_size, target_classes=target_classes,
        lambda1=lambda1, lambda2=lambda2, eta_init=lr_init,
        n_iter=n_iter, k_rank=k_rank, batch_frames=batch_frames,
        alpha=alpha, gamma=gamma, beta=beta,
        conf_threshold=tau, epsilon=epsilon,
        device=device, video_name=video_name, class_mode=class_mode,
    )
    delta = trainer.train()

    print("\n[PIPELINE] Final evaluation on TEST set (NO LEAKAGE)...")
    metrics = compute_metrics_full(
        frozen_model=frozen_model,
        frames=test_frames,    
        delta=delta,
        conf_threshold=tau,
    )

    print(f"\n  ┌─ RESULTS: {video_name} | {class_mode} {'─'*20}")
    print(f"  │  IoU_acc : {metrics['IoU_acc']:.4f}  (↓ better)")
    print(f"  │  advBR   : {metrics['advBR']:.4f}  (↓ better; 0=all vanished)")
    print(f"  │  MAP     : {metrics['MAP']:.5f} (↓ stealthier)")
    print(f"  │  ‖δ‖_*  : {metrics['nuc_norm']:.2f}")
    print(f"  │  Boxes   : {metrics['n_adv']} adv / {metrics['n_clean']} clean [on TEST]")
    print(f"  └{'─'*50}")

    save_perturbation(delta, metrics, video_name, class_mode, hyperparams, perturb_dir)
    sample = test_frames[len(test_frames)//2] if test_frames else None
    visualize_perturbation(delta, video_name, class_mode, viz_dir, sample)
    plot_training_curves(trainer.log, video_name, class_mode, viz_dir)

    return metrics

print("[OK] run_attack_pipeline (Train/Val/Test separation) defined.")

[OK] run_attack_pipeline (Train/Val/Test separation) defined.


In [20]:
# CELL 16 — UNIVERSAL MODE + BATCH RUNNER (updated)
# Universal mode: Each video contributes a number of frames proportional to its total frame count.

def run_universal_attack(class_mode: str, dataset_root: str,
                         dataset_dir: Path, perturb_dir: Path,
                         viz_dir: Path, total_pool: int = 600,
                         sample_mode: str = "uniform", **kwargs) -> Dict:
    
    target_classes = CLASS_MODE_TO_YOLO[class_mode]
    
    # Three separate pools
    all_train_frames, all_val_frames, all_test_frames = [], [], []
    total_all = sum(VIDEO_FRAME_COUNTS.values())
    
    def get_paths(txt_name: str, vname: str) -> List[str]:
        txt_path = dataset_dir / txt_name
        if not txt_path.exists(): return []
        with open(txt_path, 'r') as f:
            return [p.strip() for p in f if vname in p.strip()]
    
    print(f"[UNIVERSAL] Pooling frames (Train/Val/Test separated) → Target: {total_pool}")
    
    for vname in VIDEO_NAMES:
        ratio = VIDEO_FRAME_COUNTS[vname] / total_all
        v_budget = max(30, int(total_pool * ratio))
        
        v_train = get_paths("train.txt", vname)
        v_val   = get_paths("val.txt",   vname)
        v_test  = get_paths("test.txt",  vname)
        
        all_train_frames.extend(load_frames_from_path_list(
            v_train, kwargs.get("img_size", IMG_SIZE), v_budget,
            kwargs.get("device", DEVICE), sample_mode))
        all_val_frames.extend(load_frames_from_path_list(
            v_val, kwargs.get("img_size", IMG_SIZE), min(v_budget, 50),
            kwargs.get("device", DEVICE), "uniform"))
        all_test_frames.extend(load_frames_from_path_list(
            v_test, kwargs.get("img_size", IMG_SIZE), min(v_budget, 50),
            kwargs.get("device", DEVICE), "uniform"))
        
        print(f"  {vname:<30} → train:{len(v_train)} | val:{len(v_val)} | test:{len(v_test)}")

    frozen_model = FrozenYOLO(
        model_name=kwargs.get("yolo_model", YOLO_MODEL),
        device=kwargs.get("device", DEVICE),
        img_size=kwargs.get("img_size", IMG_SIZE),
        target_classes=target_classes,
        conf_threshold=kwargs.get("tau", TAU),
    )
    
    trainer = UAPTrainer(
        frozen_model=frozen_model,
        train_frames=all_train_frames,  # ← Training set
        eval_frames=all_val_frames,     # ← Validation set (for selecting the best model)
        img_size=kwargs.get("img_size", IMG_SIZE),
        target_classes=target_classes,
        lambda1=kwargs.get("lambda1", LAMBDA1),
        lambda2=kwargs.get("lambda2", LAMBDA2),
        eta_init=kwargs.get("lr_init", LR_INIT),
        n_iter=kwargs.get("n_iter", N_ITER),
        k_rank=kwargs.get("k_rank", K_RANK),
        batch_frames=kwargs.get("batch_frames", BATCH_FRAMES),
        alpha=kwargs.get("alpha", ALPHA),
        gamma=kwargs.get("gamma", GAMMA),
        beta=kwargs.get("beta", BETA),
        conf_threshold=kwargs.get("tau", TAU),
        epsilon=kwargs.get("epsilon", EPSILON),
        device=kwargs.get("device", DEVICE),
        video_name="universal", class_mode=class_mode,
    )
    delta = trainer.train()
    
    # Final evaluation on the TEST POOL (unseen data)
    metrics = compute_metrics_full(
        frozen_model, all_test_frames, delta, kwargs.get("tau", TAU)
    )
    
    save_perturbation(delta, metrics, "universal", class_mode,
                      {**kwargs, "total_pool": total_pool}, perturb_dir)
    sample = all_test_frames[0] if all_test_frames else None
    visualize_perturbation(delta, "universal", class_mode, viz_dir, sample)
    plot_training_curves(trainer.log, "universal", class_mode, viz_dir)
    
    return metrics
    
def run_all_12_uaps(dataset_root:   str  = DATASET_ROOT,
                     custom_hp:      Optional[Dict] = None,
                     skip_if_exists: bool = True) -> Dict:
    """
    Generate all 12 UAPs: 6 videos × 2 classes.
    The max_frames parameter is automatically determined from VIDEO_MAX_FRAMES (precomputed for each video).
    sample_mode="uniform" ensures full coverage across the timeline.
    """
    custom_hp = custom_hp or {}
    all_results: Dict = {}

    print("[BATCH] Preparing dataset ...")
    try:
        dataset_dir, _ = build_yolo_dataset(
            dataset_root, DATASET_DIR, TRAIN_RATIO, VAL_RATIO)
    except Exception as e:
        print(f"  [WARN] {e} — using demo dataset")
        dataset_dir = _create_demo_dataset(DATASET_DIR)

    print("\n[BATCH] Frame budget per video:")
    for v, mf in VIDEO_MAX_FRAMES.items():
        total = VIDEO_FRAME_COUNTS[v]
        print(f"  {v:<30}  {mf}/{total} frames  "
              f"(stride≈{total//mf})")

    pair_num = 0
    for vname in VIDEO_NAMES:
        for cmode in ["person", "vehicle"]:
            pair_num += 1
            key      = (vname, cmode)
            override = custom_hp.get(key, {})

            print(f"\n{'─'*60}")
            print(f"  UAP {pair_num}/12  →  {vname} | {cmode}")
            if override:
                print(f"  Override hyperparams: {override}")
            print(f"{'─'*60}")

            try:
                m = run_attack_pipeline(
                    video_name     = vname,
                    class_mode     = cmode,
                    dataset_root   = dataset_root,
                    dataset_dir    = dataset_dir,
                    perturb_dir    = PERTURB_DIR,
                    viz_dir        = VIZ_DIR,
                    # max_frames=None → automatically derived from VIDEO_MAX_FRAMES
                    max_frames     = override.get("max_frames", None),
                    sample_mode    = override.get("sample_mode", "uniform"),
                    lambda1        = override.get("lambda1",      LAMBDA1),
                    lambda2        = override.get("lambda2",      LAMBDA2),
                    epsilon        = override.get("epsilon",      EPSILON),
                    lr_init        = override.get("lr_init",      LR_INIT),
                    n_iter         = override.get("n_iter",       N_ITER),
                    k_rank         = override.get("k_rank",       K_RANK),
                    tau            = override.get("tau",          TAU),
                    alpha          = override.get("alpha",        ALPHA),
                    gamma          = override.get("gamma",        GAMMA),
                    beta           = override.get("beta",         BETA),
                    batch_frames   = override.get("batch_frames", BATCH_FRAMES),
                    skip_if_exists = skip_if_exists,
                )
                all_results[key] = m
            except Exception as e:
                print(f"  [ERROR] {e}")
                all_results[key] = {"error": str(e)}

    return all_results

print("[OK] Universal + run_all_12_uaps (stratified) defined.")

[OK] Universal + run_all_12_uaps (stratified) defined.


In [21]:
# CELL 17 — SUMMARY TABLE & APPLY UAP

def print_summary_table(all_results: Dict):
    print("\n" + "═"*72)
    print(f"{'SUMMARY — All UAP Results':^72}")
    print("═"*72)
    header = (f"{'Video':<28} {'Class':<9} {'advBR↓':>7} "
              f"{'IoU_acc↓':>9} {'MAP↓':>8} {'‖δ‖_*':>8}")
    print(header)
    print("─"*72)
    for (vname, cmode), m in sorted(all_results.items()):
        if "error" in m:
            print(f"  {vname:<26} {cmode:<9}  ERROR: {m['error'][:30]}")
        else:
            print(f"  {vname:<26} {cmode:<9} "
                  f"{m.get('advBR',0):>7.4f} "
                  f"{m.get('IoU_acc',0):>9.4f} "
                  f"{m.get('MAP',0):>8.5f} "
                  f"{m.get('nuc_norm',0):>8.2f}")
    print("═"*72)


def apply_uap_demo(perturb_dir:    Path,
                    video_name:     str,
                    class_mode:     str,
                    frame_dir:      str,
                    output_dir:     Path,
                    max_frames:     int = 30,
                    conf_threshold: float = 0.25):
    
    data  = load_perturbation(perturb_dir, video_name, class_mode)
    delta = data["delta"].to(DEVICE)

    (output_dir / "clean").mkdir(parents=True, exist_ok=True)
    (output_dir / "adversarial").mkdir(parents=True, exist_ok=True)

    yolo   = YOLO(YOLO_MODEL)
    frames = load_frames_for_video(
        frame_dir, img_size=IMG_SIZE,
        max_frames=max_frames, device=DEVICE)

    for i, x_clean in enumerate(
            tqdm(frames, desc=f"  Apply UAP ({video_name}|{class_mode})")):
        x_adv   = torch.clamp(x_clean + delta, 0.0, 1.0)
        to_np   = lambda t: (t.cpu().numpy().transpose(1,2,0)*255).astype(np.uint8)
        c_np  = cv2.cvtColor(to_np(x_clean), cv2.COLOR_RGB2BGR)
        a_np  = cv2.cvtColor(to_np(x_adv), cv2.COLOR_RGB2BGR)

        c_res   = yolo(c_np, verbose=False, conf=conf_threshold)[0]
        a_res   = yolo(a_np, verbose=False, conf=conf_threshold)[0]

        cv2.imwrite(str(output_dir/"clean"/f"frame_{i:04d}.jpg"),
                    cv2.cvtColor(c_res.plot(), cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(output_dir/"adversarial"/f"frame_{i:04d}.jpg"),
                    cv2.cvtColor(a_res.plot(), cv2.COLOR_RGB2BGR))

    print(f"  [DEMO] Frames saved → {output_dir}")

print("[OK] Summary table & apply_uap_demo defined.")

[OK] Summary table & apply_uap_demo defined.


In [ ]:
# CELL 18 — MAIN EXECUTION
print("═"*62)
print("  Structured Universal Adversarial Attacks — YOLOv11m")
print("  arXiv:2510.14460v1  |  Jacob et al. (2025)")
print("═"*62)

# ── Step 1: Chuẩn bị dataset ─────────────────────────────────────
print("\n[STEP 1] Dataset preparation ...")
try:
    dataset_dir, video_test_files = build_yolo_dataset(
        DATASET_ROOT, DATASET_DIR, TRAIN_RATIO, VAL_RATIO)
except Exception as e:
    print(f"  [WARN] {e}")
    print("  → Falling back to demo synthetic dataset ...")
    dataset_dir = _create_demo_dataset(DATASET_DIR, n_frames=40)
    video_test_files = {}

# ── Step 2: runing attack ──────────────────────────────────────────
all_results: Dict = {}

if VIDEO_TARGET == "universal":
    print("\n[STEP 2] Universal attack — all 6 videos pooled")
    for cmode in (["person","vehicle"] if CLASS_MODE == "both"
                   else [CLASS_MODE]):
        m = run_universal_attack(
            class_mode   = cmode,
            dataset_root = DATASET_ROOT,
            dataset_dir  = dataset_dir,
            perturb_dir  = PERTURB_DIR,
            viz_dir      = VIZ_DIR,
            lambda1=LAMBDA1, lambda2=LAMBDA2, epsilon=EPSILON,
            lr_init=LR_INIT, n_iter=N_ITER, k_rank=K_RANK,
            tau=TAU, alpha=ALPHA, gamma=GAMMA, beta=BETA,
            batch_frames=BATCH_FRAMES, max_frames=MAX_FRAMES,
            img_size=IMG_SIZE, device=DEVICE,
        )
        all_results[("universal", cmode)] = m

else:
    print(f"\n[STEP 2] Single attack: {VIDEO_TARGET} | {CLASS_MODE}")
    m = run_attack_pipeline(
        video_name     = VIDEO_TARGET,
        class_mode     = CLASS_MODE,
        dataset_root   = DATASET_ROOT,
        dataset_dir    = dataset_dir,
        perturb_dir    = PERTURB_DIR,
        viz_dir        = VIZ_DIR,
        lambda1        = LAMBDA1,
        lambda2        = LAMBDA2,
        epsilon        = EPSILON,
        lr_init        = LR_INIT,
        n_iter         = N_ITER,
        k_rank         = K_RANK,
        tau            = TAU,
        alpha          = ALPHA,
        gamma          = GAMMA,
        beta           = BETA,
        batch_frames   = BATCH_FRAMES,
        max_frames     = MAX_FRAMES,
        img_size       = IMG_SIZE,
        yolo_model     = YOLO_MODEL,
        device         = DEVICE,
        skip_if_exists = False,
    )
    all_results[(VIDEO_TARGET, CLASS_MODE)] = m

# ── Step 3: Summary ──────────────────────────────────────────────
print_summary_table(all_results)
list_saved_perturbations(PERTURB_DIR)

print(f"\n[DONE] Perturbations → {PERTURB_DIR}")
print(f"[DONE] Visualizations → {VIZ_DIR}")

══════════════════════════════════════════════════════════════
  Structured Universal Adversarial Attacks — YOLOv11m
  arXiv:2510.14460v1  |  Jacob et al. (2025)
══════════════════════════════════════════════════════════════

[STEP 1] Dataset preparation ...
  [annichkov_most_001] train=639 | val=137 | test=138
  [annichkov_most_002] train=2524 | val=541 | test=542
  [dvorzovy_most_001] train=810 | val=173 | test=175
  [gostiny_dvor_001] train=1348 | val=288 | test=290
  [gostiny_dvor_002] train=1722 | val=369 | test=370
  [zagorodny_proezd_001] train=638 | val=136 | test=138

[DATA] Done → train=7681 | val=1644 | test=1653

[STEP 2] Single attack: gostiny_dvor_001 | vehicle

[PIPELINE] gostiny_dvor_001 | vehicle
  Total frames: 1926 | max_frames: 1200
  ✅ TRAIN: 1200 frames (gradient update)
  ✅ VAL: 100 frames (pick best_delta)
  ✅ TEST: 100 frames (final report ONLY)
  [YOLO] Loaded yolo11m.pt | params=20,114,688 | target_cls={2, 5, 7}
  [YOLO] Loaded yolo11m.pt | params=20,114,688 

AO-Exp :  51%|█████     | 1017/2000 [1:17:42<1:15:54,  4.63s/it, advBR=0.079, ‖δ‖_*=352.2, MAP=0.0065, det=30/382]

In [56]:
import matplotlib.pyplot as plt
import numpy as np

def _plot_eval_statistics(results_per_frame, conf_clean, conf_adv, video_name, class_mode, save_path):
    if not results_per_frame:
        return
        
    frames = [r["frame_idx"] for r in results_per_frame]
    n_clean = [r["n_clean"] for r in results_per_frame]
    n_adv = [r["n_adv"] for r in results_per_frame]
    suppress_rate = [r["suppress_rate"] * 100 for r in results_per_frame]

    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f"Evaluation Statistics | {video_name} | {class_mode}", fontweight="bold", fontsize=12)

    # 1. Boxes detected: Clean vs Adversarial (Area chart)
    axs[0, 0].fill_between(frames, n_clean, color='#2ecc71', alpha=0.5, label='Clean')
    axs[0, 0].fill_between(frames, n_adv, color='#e74c3c', alpha=0.7, label='Adversarial')
    axs[0, 0].set_title("Boxes detected: Clean vs Adversarial", fontsize=10)
    axs[0, 0].set_ylabel("# Bounding boxes")
    axs[0, 0].set_xlabel("Frame index (timeline)")
    axs[0, 0].legend()

    # 2. Suppress rate per frame
    axs[0, 1].bar(frames, suppress_rate, color='#e74c3c', alpha=0.8)
    axs[0, 1].axhline(100, color='green', linestyle='--', alpha=0.5, label='100% = all vanished')
    axs[0, 1].set_title("Suppress rate per frame (%)", fontsize=10)
    axs[0, 1].set_ylabel("Suppress rate (%)")
    axs[0, 1].set_xlabel("Frame index (timeline)")
    axs[0, 1].legend()

    # 3. Confidence score distribution
    axs[1, 0].hist(conf_clean, bins=50, color='#2ecc71', alpha=0.5, label=f'Clean (n={len(conf_clean)})')
    axs[1, 0].hist(conf_adv, bins=50, color='#e74c3c', alpha=0.7, label=f'Adversarial (n={len(conf_adv)})')
    axs[1, 0].set_title("Confidence score distribution", fontsize=10)
    axs[1, 0].set_ylabel("Density / Count")
    axs[1, 0].set_xlabel("Confidence score")
    axs[1, 0].legend()

    # 4. Distribution of box count per frame
    axs[1, 1].hist(n_clean, bins=30, color='#2ecc71', alpha=0.5, label='Clean')
    axs[1, 1].hist(n_adv, bins=30, color='#e74c3c', alpha=0.7, label='Adversarial')
    axs[1, 1].set_title("Distribution of box count per frame", fontsize=10)
    axs[1, 1].set_ylabel("# Frames")
    axs[1, 1].set_xlabel("# Bounding boxes")
    axs[1, 1].legend()

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.close()

In [ ]:
# CELL 18b — EVALUATION ON THE TEST SET (RIGOROUS MATCHING)  
from scipy.optimize import linear_sum_assignment
from torchvision.ops import box_iou

def evaluate_uap_on_test_set(
    video_name: str, class_mode: str, dataset_dir: Path, perturb_dir: Path,
    output_dir: Path, yolo_model: str = YOLO_MODEL, img_size: int = IMG_SIZE,
    conf_threshold: float = TAU, iou_threshold: float = 0.5,
    device: torch.device = DEVICE, save_annotated: bool = True,
    max_viz_frames: int = 30
) -> Dict:
    
    print("\n" + "█"*62)
    print(f"  EVALUATION ON THE TEST SET (Hungarian IoU Matching ≥ 0.5)")
    print(f"  video={video_name} | class={class_mode}")
    print("█"*62)
    
    pert_path = perturb_dir / f"uap_{video_name}_{class_mode}.pt"
    if not pert_path.exists():
        print(f"[ERROR] Not found: {pert_path}"); return {}
        
    payload = torch.load(str(pert_path), map_location=device, weights_only=False)
    delta = payload["delta"].to(device)
    print(f"\n[LOAD] {pert_path.name}")
    
    test_txt = dataset_dir / "test.txt"
    with open(test_txt) as f:
        video_test_paths = sorted([l.strip() for l in f if video_name in l.strip()], key=lambda p: Path(p).name)
    print(f"[TEST] {len(video_test_paths)} frames loaded.")
    
    target_classes = CLASS_MODE_TO_YOLO[class_mode]
    frozen_model = FrozenYOLO(model_name=yolo_model, device=device, img_size=img_size,
                              target_classes=target_classes, conf_threshold=conf_threshold)
    
    out_base = output_dir / f"eval_{video_name}_{class_mode}"
    if save_annotated:
        for d in [out_base/"clean", out_base/"adversarial", out_base/"diff"]: d.mkdir(parents=True, exist_ok=True)
    viz_indices = set(VideoFrameDataset._uniform_indices(len(video_test_paths), min(max_viz_frames, len(video_test_paths)))) if save_annotated else set()
    
    print(f"\n[EVAL] Processing frames with rigorous matching...")
    results_per_frame = []
    total_nc, total_na, total_iou_sum = 0, 0, 0.0
    frames_attacked, frames_all_vanish, frames_no_object = 0, 0, 0
    conf_clean_list, conf_adv_list = [], []
    t_start = time.time()
    
    for i, img_path in enumerate(tqdm(video_test_paths, desc="  Evaluating")):
        img_bgr = cv2.imread(img_path)
        if img_bgr is None: continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_res = cv2.resize(img_rgb, (img_size, img_size))
        x_clean = torch.from_numpy(img_res).permute(2,0,1).float() / 255.0
        x_clean = x_clean.to(device)
        x_adv   = torch.clamp(x_clean + delta, 0.0, 1.0)
        
        clean_boxes = frozen_model.get_boxes(x_clean.unsqueeze(0), conf_threshold)
        adv_boxes   = frozen_model.get_boxes(x_adv.unsqueeze(0), conf_threshold)
        nc, na = len(clean_boxes), len(adv_boxes)
        total_nc += nc; total_na += na
        
        # Confidence tracking
        if clean_boxes: conf_clean_list.extend([b["conf"] for b in clean_boxes])
        if adv_boxes:   conf_adv_list.extend([b["conf"] for b in adv_boxes])
        
        matched = 0
        if nc > 0 and na > 0:
            c_boxes = torch.tensor([b['xyxy'] for b in clean_boxes])
            a_boxes = torch.tensor([b['xyxy'] for b in adv_boxes])
            iou_mat = box_iou(c_boxes, a_boxes)
            row_ind, col_ind = linear_sum_assignment(-iou_mat.cpu().numpy())
            for r, c in zip(row_ind, col_ind):
                if iou_mat[r, c] >= iou_threshold:
                    matched += 1
                    total_iou_sum += iou_mat[r, c].item()
                    
        fn = max(0, nc - matched)  # False Negatives: Clean boxes that are missed or mislocalized
        if nc == 0: frames_no_object += 1
        else:
            if fn > 0: frames_attacked += 1
            if na == 0: frames_all_vanish += 1
            
        results_per_frame.append({
            "frame_idx": i, "frame_name": Path(img_path).name,
            "n_clean": nc, "n_adv": na,
            "suppressed": fn,
            "suppress_rate": fn / max(nc, 1),
            "all_vanished": (nc > 0 and na == 0),
            "conf_clean": float(np.mean([b["conf"] for b in clean_boxes])) if clean_boxes else 0.0,
            "conf_adv": float(np.mean([b["conf"] for b in adv_boxes])) if adv_boxes else 0.0
        })
        
        # Save annotated visualizations (retaining the original drawing logic)
        if save_annotated and i in viz_indices:
            def draw_boxes(img, boxes, color):
                out = img.copy()
                for b in boxes:
                    x1,y1,x2,y2=[int(v) for v in b["xyxy"]]
                    cv2.rectangle(out,(x1,y1),(x2,y2), color, 2)
                    cv2.putText(out, f"cls{b['cls']} {b['conf']:.2f}", (x1, max(y1-4,0)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
                return out
            ann_c = draw_boxes(img_res, clean_boxes, (0,200,0))
            adv_np = (x_adv.cpu().numpy().transpose(1,2,0)*255).astype(np.uint8)
            ann_a = draw_boxes(adv_np, adv_boxes, (0,0,255))
            
            diff = np.abs(img_res.astype(np.float32) - adv_np.astype(np.float32))
            diff = (diff / max(diff.max(), 1e-6) * 255).astype(np.uint8)
            cv2.imwrite(str(out_base/"clean"/f"frame_{i:04d}.jpg"), cv2.cvtColor(ann_c, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(out_base/"adversarial"/f"frame_{i:04d}.jpg"), cv2.cvtColor(ann_a, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(out_base/"diff"/f"frame_{i:04d}.jpg"), cv2.applyColorMap(diff.max(axis=2), cv2.COLORMAP_HOT))

    # Aggregate final metrics
    T = len(results_per_frame)
    adv_br = total_na / max(total_nc, 1)
    iou_acc = total_iou_sum / max(T, 1)
    map_val = delta.abs().mean().item()
    nuc_norm = sum(torch.linalg.svdvals(delta[c]).sum().item() for c in range(delta.shape[0]))
    frames_with_obj = T - frames_no_object
    attack_rate = frames_attacked / max(frames_with_obj, 1)
    vanish_rate = frames_all_vanish / max(frames_with_obj, 1)
    mean_conf_clean = float(np.mean(conf_clean_list)) if conf_clean_list else 0.0
    mean_conf_adv = float(np.mean(conf_adv_list)) if conf_adv_list else 0.0
    conf_drop_pct = (1 - mean_conf_adv/max(mean_conf_clean, 1e-8)) * 100
    invisible_pct = (delta.abs() < 8/255).float().mean().item() * 100
    
    # Print results (retaining the original format)
    print("\n" + "="*62)
    print(f"  TEST SET RESULTS (Hungarian IoU Matching)")
    print(f"  video={video_name} | class={class_mode}")
    print("="*62)
    print(f"""
📊 DETECTION (Actual bounding boxes after NMS)
Test frames evaluated : {T}
Frames with objects (clean): {frames_with_obj} ({frames_with_obj/max(T,1)*100:.1f}%)
Frames without objects : {frames_no_object} ({frames_no_object/max(T,1)*100:.1f}%)
Total CLEAN boxes (after NMS): {total_nc}
Total ADV boxes (after NMS) : {total_na}
Suppressed boxes : {total_nc-total_na} ({(1-adv_br)*100:.1f}%)

🎯 ATTACK EFFECTIVENESS
advBR   : {adv_br:.4f} (↓ better | 0=all vanished)
IoU_acc : {iou_acc:.4f} (↓ better)
Attack rate : {frames_attacked}/{frames_with_obj} = {attack_rate*100:.1f}%
Vanish rate : {frames_all_vanish}/{frames_with_obj} = {vanish_rate*100:.1f}%

🔍 CONFIDENCE ANALYSIS
Mean conf CLEAN: {mean_conf_clean:.4f} | Mean conf ADV: {mean_conf_adv:.4f} | Conf drop: {conf_drop_pct:.1f}%

🔬 PERTURBATION QUALITY
MAP(Mean Abs) : {map_val:.6f} (↓ stealthier)
Nuclear norm  : {nuc_norm:.4f}
||δ||<8/255(invis): {invisible_pct:.1f}% pixels
""")
    
    if save_annotated: print(f"  📁 Annotated: {out_base}")
    _plot_eval_statistics(results_per_frame, conf_clean_list, conf_adv_list, video_name, class_mode, save_path=out_base/"statistics.png")
    
    return {
        "video_name": video_name, "class_mode": class_mode,
        "n_test_frames": T, "frames_with_obj": frames_with_obj,
        "total_clean": total_nc, "total_adv": total_na,
        "suppressed": total_nc - total_na,
        "advBR": round(adv_br, 6), "IoU_acc": round(iou_acc, 6),
        "attack_rate": round(attack_rate, 4), "vanish_rate": round(vanish_rate, 4),
        "mean_conf_clean": round(mean_conf_clean, 4), "mean_conf_adv": round(mean_conf_adv, 4),
        "conf_drop_pct": round(conf_drop_pct, 2), "MAP": round(map_val, 6),
        "nuc_norm": round(nuc_norm, 4), "invisible_pct": round(invisible_pct, 2),
        "per_frame": results_per_frame
    }

# Execute immediately
print("█"*62)
print("  POST-TRAINING EVALUATION (RIGOROUS METRICS)")
print("█"*62)
test_eval_results = evaluate_uap_on_test_set(
    video_name=VIDEO_TARGET, class_mode=CLASS_MODE, dataset_dir=dataset_dir,
    perturb_dir=PERTURB_DIR, output_dir=OUTPUT_DIR, yolo_model=YOLO_MODEL,
    img_size=IMG_SIZE, conf_threshold=TAU, iou_threshold=0.5,
    device=DEVICE, save_annotated=True, max_viz_frames=30
)